In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate


In [4]:
loader = PyPDFLoader("../data/India_Overview_5_Pages_for_RAG.pdf")
docs = loader.load()
len(docs)

6

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap = 200)
splitted_data= splitter.split_documents(docs)
len(splitted_data)

15

In [6]:
embiddings = GoogleGenerativeAIEmbeddings(model= "gemini-embedding-2-preview")


In [7]:
vector_store = Chroma.from_documents(
    documents= splitted_data,
    embedding= embiddings
)

In [13]:
query = "Give me some information on India"
data = vector_store.similarity_search(query=query)
print(data)

[Document(id='552540e0-aa32-4a93-a75c-64f3305dec88', metadata={'creationdate': '2026-09-13T14:45:05+00:00', 'creator': '(unspecified)', 'source': '../data/India_Overview_5_Pages_for_RAG.pdf', 'total_pages': 6, 'title': '(anonymous)', 'author': '(anonymous)', 'keywords': '', 'page': 0, 'trapped': '/False', 'page_label': '1', 'producer': 'ReportLab PDF Library - (opensource)', 'moddate': '2026-09-13T14:45:05+00:00', 'subject': '(unspecified)'}, page_content="India Overview — Page 1\n India\n A 5-Page Overview for PDF, Embedding & RAG Testing\nIntroduction\nIndia is a large and diverse country in South Asia with a long history, varied geography, many languages,\nand a rich cultural heritage. The country combines ancient traditions with modern cities, a growing\ntechnology sector, a large agricultural economy, and a parliamentary democratic system.\nGeography\nIndia extends from the Himalayan mountain system in the north to the Indian Ocean in the south. The\nArabian Sea lies to the west a

In [ ]:
print(data[0].page_content)

In [ ]:
context =""
for doc in data:
    context += doc.page_content + "\n"
print(context)

In [16]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")


In [ ]:
res = llm.invoke(f"""Can you provide me the asnwer based on the provided context for my question, context:{context} and question:{query} """)
print(res.content[0]["text"])

### Chain - context_generate | prompt | llm | stringparser

In [23]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
        print(context)

    return{
        "context": context,
        "question": query
    }
    

In [24]:
prompt = PromptTemplate.from_template(""" 
    You are a assistant and provide the context on the basis of question and if you dont know the asnwer simply say 'I don't know'
    Context: {context}
    Question:{question}    

""")

In [25]:
rag_chain = get_context | prompt | llm

In [ ]:
result = rag_chain.invoke("What is the Independence day of china?")
print(result.content[0]["text"])

I don't know.
